In [ ]:
# Step 1: Install required packages
# This will install the libraries needed for processing GEO datasets
!pip install GEOparse pandas numpy matplotlib seaborn

In [ ]:
# Step 2: Import necessary libraries
import pandas as pd
import numpy as np
import GEOparse
import matplotlib.pyplot as plt
import seaborn as sns
import os
from google.colab import files

In [ ]:
#%% Step 3: Download dataset
def download_geo_dataset(geo_accession):
    print(f"Downloading {geo_accession} dataset...")
    gse = GEOparse.get_GEO(geo=geo_accession, destdir="./")
    print("Dataset downloaded successfully!")
    return gse

gse = download_geo_dataset("GSE61635")
print("Dataset downloaded successfully!")

02-Jul-2025 15:22:54 DEBUG utils - Directory ./ already exists. Skipping.
DEBUG:GEOparse:Directory ./ already exists. Skipping.
02-Jul-2025 15:22:54 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE61nnn/GSE61635/soft/GSE61635_family.soft.gz to ./GSE61635_family.soft.gz
INFO:GEOparse:Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE61nnn/GSE61635/soft/GSE61635_family.soft.gz to ./GSE61635_family.soft.gz


100%|██████████| 53.9M/53.9M [00:01<00:00, 29.5MB/s]
02-Jul-2025 15:22:57 DEBUG downloader - Size validation passed
DEBUG:GEOparse:Size validation passed
02-Jul-2025 15:22:57 DEBUG downloader - Moving /tmp/tmp7limhk5r to /content/GSE61635_family.soft.gz
DEBUG:GEOparse:Moving /tmp/tmp7limhk5r to /content/GSE61635_family.soft.gz
02-Jul-2025 15:22:57 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE61nnn/GSE61635/soft/GSE61635_family.soft.gz
DEBUG:GEOparse:Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE61nnn/GSE61635/soft/GSE61635_family.soft.gz
02-Jul-2025 15:22:57 INFO GEOparse - Parsing ./GSE61635_family.soft.gz: 
INFO:GEOparse:Parsing ./GSE61635_family.soft.gz: 
02-Jul-2025 15:22:57 DEBUG GEOparse - DATABASE: GeoMiame
DEBUG:GEOparse:DATABASE: GeoMiame
02-Jul-2025 15:22:57 DEBUG GEOparse - SERIES: GSE61635
DEBUG:GEOparse:SERIES: GSE61635
02-Jul-2025 15:22:57 DEBUG GEOparse - PLATFORM: GPL570
DEBUG:GEOparse:PLATFORM: GPL570
/usr/local

Dataset downloaded successfully!
Dataset downloaded successfully!


In [ ]:
# Step 4: Extract the expression data
print("\nStep 4: Extracting expression data...")
# Get the first platform
gpl = list(gse.gpls.keys())[0]
# Get expression data from the dataset
expression_set = pd.DataFrame()

# Process each sample
for gsm_name, gsm in gse.gsms.items():
    # Get sample data for this GSM
    sample_data = gsm.table[['ID_REF', 'VALUE']]
    # Rename the VALUE column to the GSM ID
    sample_data.rename(columns={'VALUE': gsm_name}, inplace=True)

    # If this is the first sample, use it as the base dataframe
    if expression_set.empty:
        expression_set = sample_data
    else:
        # Merge with existing data
        expression_set = pd.merge(expression_set, sample_data, on='ID_REF', how='outer')

# Rename ID_REF to ID for consistency with R code
expression_set.rename(columns={'ID_REF': 'ID'}, inplace=True)

# Display the first few rows of the expression set
print("Expression data shape:", expression_set.shape)
print("Expression data preview:")
display(expression_set.head())


Step 4: Extracting expression data...
Expression data shape: (54675, 130)
Expression data preview:


,ID,GSM1509610,GSM1509611,GSM1509612,GSM1509613,GSM1509614,GSM1509615,GSM1509616,GSM1509617,GSM1509618,...,GSM1509729,GSM1509730,GSM1509731,GSM1509732,GSM1509733,GSM1509734,GSM1509735,GSM1509736,GSM1509737,GSM1509738
0,1007_s_at,5.55954,5.67649,6.03819,4.84100,4.79884,5.50040,4.81925,5.20313,5.13236,...,6.31955,6.39302,5.82703,5.84881,5.96819,5.99191,7.08150,5.84240,5.85513,6.15563
1,1053_at,8.05494,7.77186,8.26397,8.52121,8.17593,8.27977,7.82892,8.34298,8.05200,...,7.82093,8.11438,8.11192,8.10934,8.18908,8.07455,8.07724,8.21857,8.19755,8.09801
2,117_at,10.76600,10.97240,11.13560,11.03940,11.14590,11.32160,11.32110,11.56360,11.14170,...,9.59697,10.10950,10.45750,9.88891,9.99100,10.59020,9.92450,9.82965,10.20340,10.20430
3,121_at,5.61376,5.32898,5.40363,5.79344,5.88165,5.29313,5.78142,5.55080,5.94051,...,5.06645,5.51563,5.56972,5.89749,5.74314,5.40228,5.86330,5.81050,5.57930,6.12634
4,1255_g_at,2.50498,2.44119,2.39804,2.41362,2.53438,2.38504,2.52886,2.41573,2.85540,...,2.44492,2.40181,2.51716,2.48235,2.51950,2.45063,2.42024,2.41618,2.44117,2.32672


In [ ]:
# @title Default title text
# Step 5: Get feature data (probe annotations)
print("\nStep 5: Getting feature data...")
platform_id = list(gse.gpls.keys())[0]
feature_data = gse.gpls[platform_id].table

# Display the feature data columns to verify structure
print("Feature data columns:", feature_data.columns.tolist())
print("Feature data preview:")
display(feature_data.head())


Step 5: Getting feature data...
Feature data columns: ['ID', 'GB_ACC', 'SPOT_ID', 'Species Scientific Name', 'Annotation Date', 'Sequence Type', 'Sequence Source', 'Target Description', 'Representative Public ID', 'Gene Title', 'Gene Symbol', 'ENTREZ_GENE_ID', 'RefSeq Transcript ID', 'Gene Ontology Biological Process', 'Gene Ontology Cellular Component', 'Gene Ontology Molecular Function']
Feature data preview:


,ID,GB_ACC,SPOT_ID,Species Scientific Name,Annotation Date,Sequence Type,Sequence Source,Target Description,Representative Public ID,Gene Title,Gene Symbol,ENTREZ_GENE_ID,RefSeq Transcript ID,Gene Ontology Biological Process,Gene Ontology Cellular Component,Gene Ontology Molecular Function
0,1007_s_at,U48705,NaN,Homo sapiens,"Oct 6, 2014",Exemplar sequence,Affymetrix Proprietary Database,U48705 /FEATURE=mRNA /DEFINITION=HSU48705 Huma...,U48705,discoidin domain receptor tyrosine kinase 1 //...,DDR1 /// MIR4640,780 /// 100616237,NM_001202521 /// NM_001202522 /// NM_001202523...,0001558 // regulation of cell growth // inferr...,0005576 // extracellular region // inferred fr...,0000166 // nucleotide binding // inferred from...
1,1053_at,M87338,NaN,Homo sapiens,"Oct 6, 2014",Exemplar sequence,GenBank,M87338 /FEATURE= /DEFINITION=HUMA1SBU Human re...,M87338,"replication factor C (activator 1) 2, 40kDa",RFC2,5982,NM_001278791 /// NM_001278792 /// NM_001278793...,0000278 // mitotic cell cycle // traceable aut...,0005634 // nucleus // inferred from electronic...,0000166 // nucleotide binding // inferred from...
2,117_at,X51757,NaN,Homo sapiens,"Oct 6, 2014",Exemplar sequence,Affymetrix Proprietary Database,X51757 /FEATURE=cds /DEFINITION=HSP70B Human h...,X51757,heat shock 70kDa protein 6 (HSP70B'),HSPA6,3310,NM_002155,0000902 // cell morphogenesis // inferred from...,0005737 // cytoplasm // inferred from direct a...,0000166 // nucleotide binding // inferred from...
3,121_at,X69699,NaN,Homo sapiens,"Oct 6, 2014",Exemplar sequence,GenBank,X69699 /FEATURE= /DEFINITION=HSPAX8A H.sapiens...,X69699,paired box 8,PAX8,7849,NM_003466 /// NM_013951 /// NM_013952 /// NM_0...,0001655 // urogenital system development // in...,0005634 // nucleus // inferred from direct ass...,0000979 // RNA polymerase II core promoter seq...
4,1255_g_at,L36861,NaN,Homo sapiens,"Oct 6, 2014",Exemplar sequence,Affymetrix Proprietary Database,L36861 /FEATURE=expanded_cds /DEFINITION=HUMGC...,L36861,guanylate cyclase activator 1A (retina),GUCA1A,2978,NM_000409 /// XM_006715073,0007165 // signal transduction // non-traceabl...,0001750 // photoreceptor outer segment // infe...,0005509 // calcium ion binding // inferred fro...


In [ ]:
# Step 6: Check for correct column names for gene symbol
print("\nStep 6: Checking column names for gene symbol...")
possible_gene_symbol_cols = [
    'gene_symbol', 'Gene symbol', 'Gene Symbol', 'Symbol', 'SYMBOL',
    'gene symbol', 'GENE_SYMBOL', 'GeneName', 'Gene_Symbol', 'GENE_NAME'
]
gene_symbol_col = None

for col in possible_gene_symbol_cols:
    if col in feature_data.columns:
        gene_symbol_col = col
        print(f"Found gene symbol column: {col}")
        break

if gene_symbol_col is None:
    print("Could not find standard gene symbol column. Available columns are:", feature_data.columns.tolist())
    # Try to determine the most likely gene symbol column
    for col in feature_data.columns:
        if ('gene' in col.lower() and 'symbol' in col.lower()) or \
           ('gene' in col.lower() and 'name' in col.lower()):
            gene_symbol_col = col
            print(f"Using column as gene symbol: {col}")
            break

    if gene_symbol_col is None:
        print("Setting gene symbol to ID column as fallback")
        gene_symbol_col = 'ID'  # Fallback to ID if no symbol column found


Step 6: Checking column names for gene symbol...
Found gene symbol column: Gene Symbol


In [ ]:
# Step 6: Create gene_info dataframe
print("\nStep 6: Creating gene information table...")
# Check if the required columns exist
# The original code incorrectly assumes 'Gene symbol' or 'Gene Symbol' as column names
# Inspecting the feature_data DataFrame shows 'Symbol' is the correct column name
gene_symbol_col = 'Gene Symbol' # Correct column name in feature_data

# Create gene_info dataframe
gene_info = pd.DataFrame({
    'probe_id': feature_data['ID'],
    'gene_symbol': feature_data[gene_symbol_col],  # Use the correct column name
    'gene_id': feature_data['ID']
})

# Clean up gene symbols - remove extra information
gene_info['gene_symbol'] = gene_info['gene_symbol'].astype(str).apply(lambda x: x.split(' /// ')[0])

# Remove rows with missing gene symbols or gene IDs
gene_info = gene_info[
    (~gene_info['gene_symbol'].isna()) &
    (gene_info['gene_symbol'] != '') &
    (~gene_info['gene_id'].isna()) &
    (gene_info['gene_id'] != '')
]

print("Gene info shape:", gene_info.shape)
print("Gene info preview:")
display(gene_info.head())


Step 6: Creating gene information table...
Gene info shape: (54675, 3)
Gene info preview:


,probe_id,gene_symbol,gene_id
0,1007_s_at,DDR1,1007_s_at
1,1053_at,RFC2,1053_at
2,117_at,HSPA6,117_at
3,121_at,PAX8,121_at
4,1255_g_at,GUCA1A,1255_g_at


In [ ]:
# Create phenotype dataframe with full metadata
pheno_data = pd.DataFrame([
    {
        "sample_id": gsm.name,
        **gsm.metadata,
        **{f"characteristics_{i}": char for i, char in enumerate(gsm.metadata.get("characteristics_ch1", []))}
    }
    for gsm in gse.gsms.values() # Changed 'samples' to 'gse.gsms'
])

# Optional: Flatten lists to string if needed
pheno_data = pheno_data.applymap(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

# Reset index and clean up
pheno_data.reset_index(drop=True, inplace=True)

print("Phenotype data shape:", pheno_data.shape)
print("Phenotype data preview:")
display(pheno_data.head())

# Save the dataframe to a CSV file before attempting to download it
pheno_data.to_csv('pheno_data.csv', index=False)  # Save to CSV

files.download('pheno_data.csv') # Now you can download the file

Phenotype data shape: (129, 36)
Phenotype data preview:


/tmp/ipython-input-54-2574625092.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pheno_data = pheno_data.applymap(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)


,sample_id,title,geo_accession,status,submission_date,last_update_date,type,channel_count,source_name_ch1,organism_ch1,...,contact_address,contact_city,contact_state,contact_zip/postal_code,contact_country,supplementary_file,series_id,data_row_count,characteristics_0,characteristics_1
0,GSM1509610,"SLE patient, 1001V1",GSM1509610,Public on Jan 01 2015,Sep 22 2014,Jan 01 2015,RNA,1,SLE patient blood,Homo sapiens,...,Lilly Corporate Center,Indianapolis,IN,46285,USA,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1509...,GSE61635,54675,diagnosis: systemic lupus erythematosus (SLE),tissue: blood
1,GSM1509611,"SLE patient, 1001V2",GSM1509611,Public on Jan 01 2015,Sep 22 2014,Jan 01 2015,RNA,1,SLE patient blood,Homo sapiens,...,Lilly Corporate Center,Indianapolis,IN,46285,USA,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1509...,GSE61635,54675,diagnosis: systemic lupus erythematosus (SLE),tissue: blood
2,GSM1509612,"SLE patient, 1004V1",GSM1509612,Public on Jan 01 2015,Sep 22 2014,Jan 01 2015,RNA,1,SLE patient blood,Homo sapiens,...,Lilly Corporate Center,Indianapolis,IN,46285,USA,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1509...,GSE61635,54675,diagnosis: systemic lupus erythematosus (SLE),tissue: blood
3,GSM1509613,"SLE patient, 1005V1",GSM1509613,Public on Jan 01 2015,Sep 22 2014,Jan 01 2015,RNA,1,SLE patient blood,Homo sapiens,...,Lilly Corporate Center,Indianapolis,IN,46285,USA,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1509...,GSE61635,54675,diagnosis: systemic lupus erythematosus (SLE),tissue: blood
4,GSM1509614,"SLE patient, 1011V1",GSM1509614,Public on Jan 01 2015,Sep 22 2014,Jan 01 2015,RNA,1,SLE patient blood,Homo sapiens,...,Lilly Corporate Center,Indianapolis,IN,46285,USA,ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1509...,GSE61635,54675,diagnosis: systemic lupus erythematosus (SLE),tissue: blood


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print("\nStep 7: Processing sample information from `pheno_data`...")

# Use the sample_id and try to use 'characteristics_1' or fallback to 'title'
sample_info = pd.DataFrame({
    'geo_accession': pheno_data['sample_id'],
    'description': pheno_data.apply(
                lambda row: row['characteristics_0'] if 'characteristics_0' in row and pd.notna(row['characteristics_0']) else None,
        axis=1
    )
})

print("Sample info preview:")
display(sample_info.head())


Step 7: Processing sample information from `pheno_data`...
Sample info preview:


,geo_accession,description
0,GSM1509610,diagnosis: systemic lupus erythematosus (SLE)
1,GSM1509611,diagnosis: systemic lupus erythematosus (SLE)
2,GSM1509612,diagnosis: systemic lupus erythematosus (SLE)
3,GSM1509613,diagnosis: systemic lupus erythematosus (SLE)
4,GSM1509614,diagnosis: systemic lupus erythematosus (SLE)


In [ ]:
# Step 8: Create a mapping between GSM IDs and descriptions
print("\nStep 8: Creating sample description mapping...")
# Create a mapping dictionary
desc_mapping = dict(zip(sample_info['geo_accession'], sample_info['description']))
print("Description mapping (first 5 items):")
for i, (key, value) in enumerate(desc_mapping.items()):
    print(f"{key}: {value}")
    if i >= 4:  # Only show first 5
        break


Step 8: Creating sample description mapping...
Description mapping (first 5 items):
GSM1509610: diagnosis: systemic lupus erythematosus (SLE)
GSM1509611: diagnosis: systemic lupus erythematosus (SLE)
GSM1509612: diagnosis: systemic lupus erythematosus (SLE)
GSM1509613: diagnosis: systemic lupus erythematosus (SLE)
GSM1509614: diagnosis: systemic lupus erythematosus (SLE)


In [ ]:
# Step 9: Replace GSM IDs with descriptions in expression_set
print("\nStep 9: Replacing GSM IDs with descriptions...")
original_cols = expression_set.columns.tolist()
new_cols = []

for col in original_cols:
    if col == 'ID':
        new_cols.append(col)
    else:
        # Replace with description if available
        new_cols.append(desc_mapping.get(col, col))

# Create the column mapping dictionary
col_mapping = dict(zip(original_cols, new_cols))
expression_set.rename(columns=col_mapping, inplace=True)

print("Updated column names (all):")
print(expression_set.columns.tolist()[:10])

# Save the DataFrame to a CSV file before downloading
expression_set.to_csv('expression_set.csv', index=False)

# Now download the saved file
#files.download('expression_set.csv')


Step 9: Replacing GSM IDs with descriptions...
Updated column names (all):
['ID', 'diagnosis: systemic lupus erythematosus (SLE)', 'diagnosis: systemic lupus erythematosus (SLE)', 'diagnosis: systemic lupus erythematosus (SLE)', 'diagnosis: systemic lupus erythematosus (SLE)', 'diagnosis: systemic lupus erythematosus (SLE)', 'diagnosis: systemic lupus erythematosus (SLE)', 'diagnosis: systemic lupus erythematosus (SLE)', 'diagnosis: systemic lupus erythematosus (SLE)', 'diagnosis: systemic lupus erythematosus (SLE)']


In [ ]:
# prompt: check the chosen column and give me every unique name in list

# Replace 'characteristics_0' with the actual column name you want to check
column_name = 'characteristics_0'

if column_name in pheno_data.columns:
    unique_names = pheno_data[column_name].unique().tolist()
    print(f"Unique values in column '{column_name}':")
    print(unique_names)
else:
    print(f"Column '{column_name}' not found in the dataframe.")
    print("Available columns are:", pheno_data.columns.tolist())

Unique values in column 'characteristics_0':
['diagnosis: systemic lupus erythematosus (SLE)', 'diagnosis: healthy']


In [ ]:
expression_set.drop(columns=["tissue: Whole blood"  ], inplace=True)
expression_set.to_csv('expression_set_updated.csv', index=False)
#files.download('expression_set_updated.csv')  # Works in Google Colab

In [ ]:
# Rename specific columns
expression_set.rename(columns={
    #"disease state: asthma severe": "disease",
    "diagnosis: systemic lupus erythematosus (SLE)" : "disease",
    "diagnosis: healthy": "control"
}, inplace=True)

# Verify remaining columns (optional check)
print("Remaining columns:", [col for col in expression_set.columns if "phenotype" in col])

# Save and download
expression_set.to_csv('expression_set_renamed.csv', index=False)
#files.download('expression_set_renamed.csv')  # Google Colab command

Remaining columns: []


In [ ]:
# Step 13: Merge expression data with gene information
print("\nStep 13: Merging datasets...")
final_data = pd.merge(expression_set, gene_info, left_on='ID', right_on='probe_id', how='left')

# Save merged data
final_data.to_csv('final_data.csv', index=False)

# Download the CSV (Colab only)
#from google.colab import files
#files.download('final_data.csv')


Step 13: Merging datasets...


In [ ]:
# prompt: final_data.csv contain ID, probe_id, gene_symbol, gene_id. i want replace gene_symbol in 1st column and delete other column which is mentioned.

import pandas as pd
from google.colab import files

# Load the final_data.csv file
final_data = pd.read_csv('final_data.csv')

# Replace the 'ID' column with 'gene_symbol'
final_data['ID'] = final_data['gene_symbol']

# Drop the unnecessary columns
final_data.drop(columns=['probe_id', 'gene_id'], inplace=True)

# Save the modified DataFrame to a new CSV file
final_data.to_csv('modified_final_data.csv', index=False)

# Download the modified CSV file (Colab only)
#files.download('modified_final_data.csv')


In [ ]:
# @title Default title text
# prompt: modified_final_data.csv analysis and number column of control and disease

import pandas as pd

# Load the modified dataset
modified_final_data = pd.read_csv('modified_final_data.csv')

# Display the first few rows to understand the structure
print(modified_final_data.head())

# Assuming 'control' and 'disease' are column names
# Check if the columns exist
if 'control' in modified_final_data.columns and 'case' in modified_final_data.columns:
    # Analyze the 'number' column (replace 'number' with the actual column name if different)
    #  If 'number' represents numerical data:
    try:
        control_number_stats = modified_final_data['control'].describe()
        disease_number_stats = modified_final_data['case'].describe()

        print("\nControl Group Statistics:")
        print(control_number_stats)

        print("\nDisease Group Statistics:")
        print(disease_number_stats)
    except TypeError:  # Handle cases where 'number' might not be numeric
        print("\nThe 'control' or 'case' columns do not appear to contain numeric data. Please verify.")
else:
    print("\nThe columns 'control' and/or 'case' were not found in the dataset. Please check column names.")

In [ ]:
import pandas as pd

# Load the modified CSV file
df = pd.read_csv('modified_final_data.csv')  # Assign modified_final_data to df

# Assuming your data is stored in a DataFrame called 'df'
# First, let's make a copy to avoid modifying the original data
df_modified = df.copy()

# Reset the index to make the current index a regular column
# (in case it's not already a regular column)
if 'gene_symbol' not in df_modified.columns:
    df_modified = df_modified.reset_index()

# Create a new DataFrame with gene_symbol as the first column
# and excluding the ID column (the original index)
columns = ['gene_symbol'] + [col for col in df_modified.columns if col != 'gene_symbol' and col != 'ID']
df_final = df_modified[columns]

# Display the first few rows to verify
print(df_final.head())

# Save to CSV
df_final.to_csv('modified_final_data.csv', index=False)

  gene_symbol   disease  disease.1  disease.2  disease.3  disease.4  \
0        DDR1   5.55954    5.67649    6.03819    4.84100    4.79884   
1        RFC2   8.05494    7.77186    8.26397    8.52121    8.17593   
2       HSPA6  10.76600   10.97240   11.13560   11.03940   11.14590   
3        PAX8   5.61376    5.32898    5.40363    5.79344    5.88165   
4      GUCA1A   2.50498    2.44119    2.39804    2.41362    2.53438   

   disease.5  disease.6  disease.7  disease.8  ...  control.20  control.21  \
0    5.50040    4.81925    5.20313    5.13236  ...     6.31955     6.39302   
1    8.27977    7.82892    8.34298    8.05200  ...     7.82093     8.11438   
2   11.32160   11.32110   11.56360   11.14170  ...     9.59697    10.10950   
3    5.29313    5.78142    5.55080    5.94051  ...     5.06645     5.51563   
4    2.38504    2.52886    2.41573    2.85540  ...     2.44492     2.40181   

   control.22  control.23  control.24  control.25  control.26  control.27  \
0     5.82703     5.84881  

In [ ]:
# Load the modified CSV file
df = pd.read_csv('modified_final_data.csv')

# ====== NEW CODE ======
# Detect rows with NaNs or blanks in ANY column
mask_nan_or_blank = (
    df.isna().any(axis=1) |  # Check for NaNs
    # Check for blanks in any element of the row (Series), remove axis=1
    df.apply(lambda row: row.astype(str).str.strip().eq('').any())
)

# Delete problematic rows
df = df[~mask_nan_or_blank]
# ======================

# Save the cleaned file
df.to_csv('cleaned_modified_final_data_GSE61635.csv', index=False)
files.download('cleaned_modified_final_data_GSE61635.csv')

/tmp/ipython-input-63-2615437168.py:13: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~mask_nan_or_blank]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>